# Vérification du téléchargement des adresses (BAN)

Ce notebook vérifie que `download.ban.download_ban` récupère les adresses de la Base Adresse Nationale (via le miroir WFS IGN `BAN.DATA.GOUV:ban`), filtrées sur le code INSEE exact de l'entité, et affiche le résultat sur une carte.

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

from core.config import load_entities
from download.limites_administratives import fetch_commune_boundary, resolve_bbox
from download.ban import download_ban

## 1. Entité testée (issue du CSV)

In [ ]:
csv_path = repo_root / "data" / "configuration" / "entites_exemple.csv"
entity = load_entities(csv_path)[0]
entity

## 2. Téléchargement des adresses BAN de la commune

In [ ]:
bbox = resolve_bbox(entity)
written = download_ban(entity.code_insee, bbox)
written

## 3. Relecture du GeoPackage écrit

In [ ]:
import geopandas as gpd

addresses = gpd.read_file(written)
print("Nombre d'adresses :", len(addresses))
addresses[["numero", "nom_voie", "code_postal", "code_insee"]].head(10)

## 4. Affichage sur une carte `leafmap`

Reprojection en EPSG:4326 uniquement pour l'affichage. Backend `folium` (HTML statique), plus fiable pour le rendu des widgets dans VS Code.

In [ ]:
import leafmap.foliumap as leafmap

boundary_wgs84 = fetch_commune_boundary(entity.code_insee).to_crs(epsg=4326)
addresses_wgs84 = addresses.to_crs(epsg=4326)

m = leafmap.Map()
m.add_gdf(
    addresses_wgs84,
    layer_name="Adresses (BAN)",
    style={"color": "blue", "fillColor": "blue", "fillOpacity": 0.6, "radius": 3},
    info_mode="on_click",
)
m.add_gdf(
    boundary_wgs84,
    layer_name="Limite communale",
    style={"color": "red", "fillOpacity": 0},
)
m.zoom_to_gdf(boundary_wgs84)
m